In [1]:
import time
import logging
import pandas as pd
from truedata import TD_hist

def fetch_truedata_history(
    username: str,
    password: str,
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    # Initialize connection
    td_hist = TD_hist(username, password)

    df_list = []
    error_list = []

    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            df = df.rename(columns={
                'timestamp': 'Date',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list


In [2]:
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
from pathlib import Path

import os

def run_momentum_strategy(universe_file: str,
                          start_date: str,
                          end_date: str,
                          top_n: int,
                          output_root: str = "Momentum_Results",
                          freeze_cutoff_date: str = "2026-05-31",
                          output_dir_override: str = None) -> str:
    """
    Run rolling-window momentum strategy on given stock universe.

    Parameters
    ----------
    universe_file : str
        Path to universe file (.csv or .xlsx) with columns ["Symbol", "ISIN Code"].
    start_date : str
        Start date for backtest (YYYY-MM-DD).
    end_date : str
        End date for backtest (YYYY-MM-DD).
    top_n : int
        Number of top ranked stocks to select each window.
    output_root : str
        Root folder where results will be saved.
    freeze_cutoff_date : str
        Date (YYYY-MM-DD) up to which existing window files will not be overwritten.
    output_dir_override : str or None
        If provided, use this exact directory to store window files instead of
        the default `<output_root>/<universe_name>_{top_n}_stocks_results`.

    Returns
    -------
    str
        Path to master summary file.
    """


    # ==== 1. Load Universe ====
    if universe_file.endswith(".csv"):
        stock_list = pd.read_csv(universe_file)[["Symbol", "ISIN Code"]]
    else:
        stock_list = pd.read_excel(universe_file)[["Symbol", "ISIN Code"]]
    # username = os.getenv("TRUEDATA_USERNAME")
    # password = os.getenv("TRUEDATA_PASSWORD")
    
    username = os.getenv("TRUEDATA_USERNAME")
    password = os.getenv("TRUEDATA_PASSWORD")
    # username = 'td105'
    # password = os.getenv("TRUEDATA_PASSWORD")
    # stock_list["Ticker"] = stock_list["Symbol"] + ".NS"
    stock_list["Ticker"] = stock_list["Symbol"].astype(str).str.strip().str.upper()
    banned_tickers = {"APARINDS"}
    stock_list = stock_list[~stock_list["Ticker"].isin(banned_tickers)].copy()
    symbol_list = stock_list["Ticker"].tolist()
    universe_name = Path(universe_file).stem

    # Determine output directory (allow override)
    if output_dir_override:
        output_dir = output_dir_override
    else:
        output_dir = os.path.join(output_root, f"{universe_name}_{top_n}_stocks_results")
    os.makedirs(output_dir, exist_ok=True)

    # ==== 2. Download Data ====
    total_start = pd.to_datetime(start_date)
    total_end = pd.to_datetime(end_date)

    print(f"\n📥 Downloading price data for {len(symbol_list)} symbols...")
    # data = yf.download(symbol_list, start=total_start.strftime('%Y-%m-%d'),
    #                    end=total_end.strftime('%Y-%m-%d'), progress=True)
    # prices_all = data["Close"]
    # prices_all.index = pd.to_datetime(prices_all.index)
    # print("✅ Download complete.")
    data, errors = fetch_truedata_history(username, password, symbol_list, duration='10 Y', bar_size='EOD')
    data['Ticker'] = data['Ticker'].astype(str).str.strip().str.upper()
    data = data[['Date', 'Close', 'Ticker']]
    data.drop_duplicates(subset=['Date', 'Ticker'], inplace=True)
    print(data)
    print("Failed tickers:", errors)
    prices = data.pivot(index="Date", columns="Ticker", values="Close")
    # optional: sort by date
    prices_all = prices.sort_index()


    # ==== 3. Create Rolling Windows ====
    windows = []
    current_start = total_start
    while True:
        current_end = current_start + relativedelta(months=6)
        if current_end > total_end:
            break
        window_prices = prices_all.loc[(prices_all.index >= current_start) & (prices_all.index < current_end)].copy()
        if not window_prices.empty:
            windows.append((current_start, current_end, window_prices))
        current_start += relativedelta(months=1)

    print(f"📊 Created {len(windows)} rolling windows.")

    # ==== 4. Process Each Window ====
    for start, end, prices in windows:
        file_suffix = f"{start.strftime('%Y%m%d')}_{end.strftime('%Y%m%d')}" 
        print(f"\n🔍 Processing window: {start.date()} → {end.date()}")

        prices.dropna(axis=1, how='all', inplace=True)
        if prices.empty:
            print("⚠️ All price data missing. Skipping window.")
            continue

        # Monthly momentum
        monthclose = prices.groupby(prices.index.strftime('%Y-%m')).tail(1)
        monthstart = prices.groupby(prices.index.strftime('%Y-%m')).head(1)
        monthstart.index = monthclose.index
        monchange = (monthclose - monthstart) / monthstart
        MOM = (monchange + 1).product() - 1
        mom = MOM * 100

        # Daily returns
        daily_ret = prices.pct_change(fill_method=None)
        positivechange = (daily_ret[daily_ret > 0].count() / daily_ret.count()) * 100
        negativechange = (daily_ret[daily_ret < 0].count() / daily_ret.count()) * 100

        result = pd.concat([positivechange, negativechange, mom], axis=1, join='inner')
        result.columns = ["Positive", "Negative", "Momentum"]
        result = result.reset_index().rename(columns={'index': 'Ticker'})
        try:
            last_prices = prices.tail(1).iloc[0].to_dict()
            last_prices = {str(k).strip().upper(): v for k, v in last_prices.items()}
        except Exception:
            last_prices = {}
        result["Last_Price"] = result["Ticker"].map(last_prices)
        pre_count = len(result)
        result = result[result["Last_Price"].isna() | (result["Last_Price"] <= 7500)]
        removed_count = pre_count - len(result)
        if removed_count:
            print(f"🔒 Excluded {removed_count} stocks with Last_Price > 7500 from this window")

        # Merge ISIN
        result = pd.merge(result, stock_list[["Ticker", "ISIN Code"]], on="Ticker", how="left")

        # Ranking
        df = result.copy()
        df["Rank_Mom"] = df["Momentum"].rank(method='min', ascending=False)
        print('rank df:', df)
        df['FIP'] = df.apply(lambda row: row['Negative'] - row['Positive'] if row['Momentum'] > 0 else np.nan, axis=1)
        # df['FIP'] = df.apply(lambda row: row['Negative'] - row['Positive'])

        df.dropna(inplace=True)
        df["FIP_rank"] = df["FIP"].rank(method="first", ascending=True)
        df["Combined_Rank"] = df["Rank_Mom"] + df["FIP_rank"]
        if end.strftime('%Y-%m-%d') == '2026-01-01':
            df = df[~(df['Ticker'].isin(['MARUTI', 'PTCIL']))]

        df = df.sort_values(by="Combined_Rank", ascending=True).head(top_n)
        df["Real_Rank"] = range(1, len(df) + 1)
        df["End_Date"] = end.strftime('%Y-%m-%d')


        # Save each window
        output_file = os.path.join(output_dir, f"momentum_{file_suffix}.xlsx")

        # Determine freeze behavior: do not overwrite existing files
        # for windows whose end date is on or before `freeze_cutoff_date`.
        try:
            freeze_cutoff = pd.to_datetime(freeze_cutoff_date).date() if freeze_cutoff_date else None
        except Exception:
            freeze_cutoff = None

        window_end_date = pd.to_datetime(end).date()

        if freeze_cutoff is not None and window_end_date <= freeze_cutoff and os.path.exists(output_file):
            print(f"⏸ Skipping overwrite of frozen window: {file_suffix} (<= {freeze_cutoff})")
        else:
            df.to_excel(output_file, index=False)
            print(f"✅ Saved results to: {output_file}")

    # ==== 5. Master File ====
    print("\n📂 Creating master summary file...")
    master_data = []
    for file in os.listdir(output_dir):
        if file.startswith("momentum_") and file.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(output_dir, file))
            selected_df = df[["End_Date", "ISIN Code", "Ticker", 'Real_Rank']].copy()

            master_data.append(selected_df)

    if not master_data:
        print("⚠️ No window files found, master not created.")
        return None

    master_df = pd.concat(master_data, ignore_index=True)
    master_file_path = os.path.join(output_dir, "master_momentum_summary.xlsx")
    master_df.to_excel(master_file_path, index=False)
    print(f"✅ Master file saved to: {master_file_path}")
    return master_file_path

In [3]:


# Momentum/Automating Momentum/Universe/Nifty_500_2025_Apr.csv
# "C:\Users\Admin\Momentum\Automating Momentum\Universe\Nifty_500_2025_Apr.csv"
master_file = run_momentum_strategy(
    universe_file=r"C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\Universe\ticker_master_may26.xlsx",
    start_date="2022-06-01",
    end_date="2026-06-01",
    top_n=20,
    output_root="Stocks_old",
    # override to append new windows into the existing Nifty_500_2025_Apr output folder
    output_dir_override=r"C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\MOMENTUM_DB_2\Stocks_old\Nifty_500_2025_Apr_20_stocks_results"
)
print("Master summary located at:", master_file)



📥 Downloading price data for 498 symbols...


(2026-05-31 19:25:56,357) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14436 Thread:21044)


2026-05-31 19:25:56,357 - WARNING - Connected successfully to TrueData Historical Data Service... 


2026-05-31 19:25:57,075 - INFO - Fetched data for 360ONE (1660 rows).


2026-05-31 19:25:58,081 - INFO - Fetched data for 3MINDIA (2475 rows).


...

🔒 Excluded 20 stocks with Last_Price > 7500 from this window
rank df:          Ticker   Positive   Negative   Momentum  Last_Price     ISIN Code  \
0        360ONE  50.000000  50.000000  39.764238      784.05  INE466L01038   
1      AARTIIND  55.737705  43.442623  60.100667      741.75  INE769A01020   
2         AAVAS  47.540984  52.459016   4.036945     1624.10  INE216P01012   
3           ABB  60.655738  39.344262  61.702157     6540.75  INE117A01022   
4     ABCAPITAL  51.639344  47.540984  30.871962      231.40  INE674K01013   
..          ...        ...        ...        ...         ...           ...   
420        ZEEL  50.000000  50.000000 -50.234197      146.95  INE256A01028   
421  ZENSARTECH  45.901639  54.098361  27.895658      613.25  INE520A01027   
422      ZENTEC  45.081967  54.098361  44.430239     1110.45  INE251B01027   
423   ZYDUSLIFE  59.836066  39.344262  67.250175      953.55  INE010B01027   
424   ZYDUSWELL  49.180328  48.360656   8.910720      333.20  INE768C010

⏸ Skipping overwrite of frozen window: 20240701_20250101 (<= 2026-05-31)

🔍 Processing window: 2024-08-01 → 2025-02-01
🔒 Excluded 19 stocks with Last_Price > 7500 from this window
rank df:          Ticker   Positive   Negative   Momentum  Last_Price     ISIN Code  \
0        360ONE  49.206349  50.793651  -9.973517     1007.75  INE466L01038   
1     AADHARHFC  46.825397  53.174603  -9.011354      398.50  INE883F01010   
2      AARTIIND  48.412698  51.587302 -42.391909      444.70  INE769A01020   
3         AAVAS  50.000000  50.000000   1.765108     1712.80  INE216P01012   
4           ABB  50.793651  48.412698 -24.964370     5874.65  INE117A01022   
..          ...        ...        ...        ...         ...           ...   
445        ZEEL  45.238095  54.761905 -29.817161      105.59  INE256A01028   
446  ZENSARTECH  49.206349  50.793651   8.584231      870.00  INE520A01027   
447      ZENTEC  53.174603  46.825397   1.727405     1741.10  INE251B01027   
448   ZYDUSLIFE  49.206349  50.

🔒 Excluded 23 stocks with Last_Price > 7500 from this window
rank df:          Ticker   Positive   Negative   Momentum  Last_Price     ISIN Code  \
0        360ONE  50.793651  48.412698  10.744098     1080.70  INE466L01038   
1     AADHARHFC  48.412698  51.587302   8.578626      509.85  INE883F01010   
2      AARTIIND  46.031746  53.174603 -10.645467      379.95  INE769A01020   
3         AAVAS  49.206349  50.000000  -4.053111     1650.00  INE216P01012   
4           ABB  53.174603  46.825397  -2.762131     5220.00  INE117A01022   
..          ...        ...        ...        ...         ...           ...   
458        ZEEL  48.412698  51.587302   1.236891      100.58  INE256A01028   
459  ZENSARTECH  50.000000  50.000000  10.775863      797.55  INE520A01027   
460      ZENTEC  47.619048  52.380952 -10.578337     1357.20  INE251B01027   
461   ZYDUSLIFE  57.142857  42.063492   9.119493      974.45  INE010B01027   
462   ZYDUSWELL  50.793651  48.412698  22.489358      476.60  INE768C010

✅ Saved results to: C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\MOMENTUM_DB_2\Stocks_old\Nifty_500_2025_Apr_20_stocks_results\momentum_20251201_20260601.xlsx

📂 Creating master summary file...


✅ Master file saved to: C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\MOMENTUM_DB_2\Stocks_old\Nifty_500_2025_Apr_20_stocks_results\master_momentum_summary.xlsx
Master summary located at: C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\MOMENTUM_DB_2\Stocks_old\Nifty_500_2025_Apr_20_stocks_results\master_momentum_summary.xlsx


In [4]:
# ----------------xxxxxxxxxxxxxxx---------------------------------------